# Medical CPT / SFT ordering + RLHF

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/s1mran/finetuning/blob/main/run_medical.ipynb)

Three runs, ~65 minutes total on a T4:

| cell | run | what it answers |
|---|---|---|
| 2 | CPT → SFT (medical) | does domain-first adaptation hold up? |
| 3 | SFT → CPT (medical) | what does reversing the order cost? |
| 4 | SFT → DPO (empathy) | can DPO align it without reward hacking? |

**Set Runtime → Change runtime type → T4 GPU before anything else.** Unsloth is
CUDA-only.

Cells 2 and 3 are a matched pair — same base model, same seed, same step counts,
only the stage order differs. That is the whole experiment; run both or neither.

## 1. Setup

~2 minutes. Needed again after every restart — Colab wipes installed packages.
No kernel restart afterwards: the scripts run as `!python` subprocesses that
pick up the new packages.

In [ ]:
!nvidia-smi

!pip install -q unsloth trl peft transformers datasets accelerate bitsandbytes

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*use_return_dict.*")
warnings.filterwarnings("ignore", message=".*max_new_tokens.*max_length.*")

ROOT = "/content/finetuning"
!git clone https://github.com/s1mran/finetuning.git {ROOT} 2>/dev/null || git -C {ROOT} pull
!git -C {ROOT} log --oneline -1

## 2. CPT → SFT (medical)

~20 min. Stage 1 continues pre-training on `epfl-llm/guidelines` (all-token
loss, LoRA r=32, embeddings unfrozen, 20% WikiText replay). Stage 2 does
response-only SFT on ChatDoctor.

**Four lines worth reading as it runs:**

1. `docs train=N eval=M -> chunks ... (no document appears on both sides)` —
   the eval split is carved at the document level, so no eval chunk overlaps a
   training chunk.
2. `[cpt-data / domain]` and `[sft-data / ...]` — real training strings. Read
   them. A wrong column or an empty context is invisible in a loss curve.
3. `[mask] supervising N/M tokens` — response-only masking is live, and should
   be a minority of tokens.
4. `[merge-check cpt-intermediate] ... (xN)` — **the important one.** Stage 2
   trains on the merged stage-1 file, not the model in memory. A ratio near
   1.00 means the merge is faithful.

`--keep-intermediate` retains `02_cpt_merged` (~270 MB) so it can be inspected
if the merge check flags.

In [ ]:
!python /content/finetuning/CPT/src/cpt_med_then_sft.py --keep-intermediate

## 3. SFT → CPT (medical) — the ablation

~20 min. Same two stages, reversed.

The hypothesis is that all-token loss on raw prose has no reason to preserve
"answer, then stop" — it pushes every position toward continuing a guideline,
including the ones right after `### Response:`. Watch `alpaca probes
terminating with EOS` across the three stage boundaries.

Worth knowing before you look: on the finance pair this prediction did **not**
hold. EOS went 0% → 100% after SFT and stayed at 100% through CPT.

In [ ]:
!python /content/finetuning/SFT/src/sft_med_then_cpt.py --keep-intermediate

## 4. SFT → DPO (RLHF)

~25 min. SFT on empathetic dialogue, then DPO on human preference pairs.

This one is empathy, not medical — there is no public preference dataset about
clinical quality, and the repo has only one RLHF experiment.

The output that matters is the **BEFORE / AFTER** table at the end. Read the
length and repetition columns *before* the accuracy column:

- accuracy up, length and repetition flat → alignment worked
- accuracy up, mean tokens climbing sharply → length exploitation
- accuracy up, repetition climbing → degeneration
- general perplexity climbing sharply → drifted off the reference

The script prints an explicit warning if mean response length grew more than
1.5×. If it fires, re-run with `--beta 0.3 --max-length-ratio 1.2`.

In [ ]:
!python /content/finetuning/RLHF/src/sft_then_dpo.py

## 5. Did it work?

`stage-1 merge` is the number to read first. Stage 2 builds on that file, and
until recently nothing measured it — which is how a merge that corrupted the
tied `embed_tokens`/`lm_head` weight stayed invisible behind a summary that
looked healthy.

In [ ]:
import json, glob, os

for p in sorted(glob.glob(f"/content/finetuning/*/reports/*/report.json")):
    r = json.loads(open(p).read())
    name = p.split("/reports/")[1].split("/")[0]
    print(f"\n{'='*66}\n{name}   [{r.get('order','?')}]\n{'='*66}")

    for k, label in (("merge_check_stage1", "stage-1 merge"),
                     ("merge_check",        "final merge")):
        m = r.get(k) or {}
        if m:
            ok = "OK" if m.get("ratio", 9) < 1.5 else "CORRUPT - do not publish"
            print(f"  {label:14}: adapter {m['adapter_ppl']:.2f} -> merged "
                  f"{m['merged_ppl']:.2f}  (x{m['ratio']:.2f})  {ok}")

    for k in ("fit_cpt", "fit_sft"):
        f = r.get(k) or {}
        if f:
            flag = "   OVERFIT" if f["eval_last"] > f["eval_first"] else ""
            print(f"  {k:14}: eval {f['eval_first']:.3f} -> {f['eval_last']:.3f} "
                  f"(best {f['eval_best']:.3f}){flag}")

    for k in ("ppl_before", "ppl_after_cpt", "ppl_after_sft", "ppl_after_dpo"):
        v = r.get(k)
        if isinstance(v, dict):
            print(f"  {k:14}: domain {v.get('domain', float('nan')):7.2f}   "
                  f"general {v.get('general', float('nan')):7.2f}")
        elif isinstance(v, (int, float)):
            print(f"  {k:14}: general {v:.2f}")

    for k in ("pref_acc_before", "pref_acc_after_sft", "pref_acc_after_dpo"):
        if k in r:
            print(f"  {k:18}: {r[k]:.3f}")

## 6. The ordering comparison

Perplexity columns should be comparable across the two orderings; the probe
behaviour is where a difference would show.

In [ ]:
import json, os

LAYOUT = {
    "CPT -> SFT": (f"/content/finetuning/CPT/reports/cpt_med_then_sft/report.json",
                   "ppl_after_cpt", "ppl_after_sft"),
    "SFT -> CPT": (f"/content/finetuning/SFT/reports/sft_med_then_cpt/report.json",
                   "ppl_after_sft", "ppl_after_cpt"),
}

for name, (path, s1, s2) in LAYOUT.items():
    if not os.path.exists(path):
        print(f"missing {path} -- run that cell first"); continue
    r = json.loads(open(path).read())
    print(f"\n{'='*66}\n{name}\n{'='*66}")
    for dom in ("domain", "general"):
        b = (r.get("ppl_before") or {}).get(dom, float("nan"))
        a = (r.get(s1) or {}).get(dom, float("nan"))
        c = (r.get(s2) or {}).get(dom, float("nan"))
        print(f"  {dom:>7} ppl: {b:7.2f} -> {a:7.2f} (stage1) -> {c:7.2f} (stage2)")
    for stage, key in (("base", "probes_base"), ("final", "probes_final")):
        rate = (r.get(key) or {}).get("eos_rate")
        if rate is not None:
            print(f"  {stage:>7} eos: {rate:.0%}")
    print("  final answers:")
    for k, v in (r.get("probes_final") or {}).items():
        if k.startswith("alpaca::"):
            print(f"    Q: {k.split('::', 1)[1]}")
            print(f"    A: {v[:200]}")

## 7. If the stage-1 merge check flagged

`embed_tokens` is tied to `lm_head` in SmolLM. Adapting it and merging can
leave the checkpoint worse than the adapter it came from. `--freeze-embeddings`
takes the embeddings out of CPT entirely — attention and MLP merge cleanly.

It costs the domain-vocabulary learning the stage exists for, so use it only
when the check says you need it. Skip this cell otherwise.

In [ ]:
!python /content/finetuning/CPT/src/cpt_med_then_sft.py --freeze-embeddings
!python /content/finetuning/SFT/src/sft_med_then_cpt.py --freeze-embeddings

## 8. Publish

Needs a **Write**-scoped token from
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens),
stored in Colab's secrets pane (🔑, left sidebar) as `HF_TOKEN`.

The upload refuses any checkpoint the merge check flagged — publishing a broken
model is worse than publishing none.

In [ ]:
from huggingface_hub import HfApi
from google.colab import userdata
import os, json

USER = "sidhusarkar"          # your HF username
token = userdata.get("HF_TOKEN")
api = HfApi()

REPOS = {
    "CPT/reports/cpt_med_then_sft": "smollm-135m-med-cpt-then-sft",
    "SFT/reports/sft_med_then_cpt": "smollm-135m-med-sft-then-cpt",
    "RLHF/reports/sft_then_dpo":    "smollm-135m-empathy-sft-then-dpo",
}

for path, name in REPOS.items():
    folder = f"/content/finetuning/{path}/04_final_merged"
    if not os.path.isdir(folder):
        print(f"skip {name}: {folder} not found"); continue
    ratio = (json.loads(open(f"/content/finetuning/{path}/report.json").read())
             .get("merge_check") or {}).get("ratio", 1.0)
    if ratio > 1.5:
        print(f"skip {name}: merge ratio {ratio:.1f}x - corrupt"); continue
    repo = f"{USER}/{name}"
    api.create_repo(repo, repo_type="model", exist_ok=True, token=token)
    api.upload_folder(folder_path=folder, repo_id=repo, repo_type="model", token=token)
    print(f"https://huggingface.co/{repo}")

## 9. Back up to Drive

Colab recycles VMs on idle, so anything under `/content` is temporary. This
keeps the adapters, merged models and `report.json` files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DEST = "/content/drive/MyDrive/finetuning_reports"
!mkdir -p "$DEST"
!cp -r /content/finetuning/CPT/reports "$DEST/CPT"
!cp -r /content/finetuning/SFT/reports "$DEST/SFT"
!cp -r /content/finetuning/RLHF/reports "$DEST/RLHF"
!du -sh "$DEST"